# Chicago Crime dataset 2001 - Present day

[Dataset (Export)](https://data.cityofchicago.org/Public-Safety/Crimes-2001-to-Present/ijzp-q8t2/data) Intialising dataset download takes a while.
 https://chicago.suntimes.com/2021/11/26/22639255/dead-end-drug-arrests-drugs-possession-chicago
 

Dataset contains roughly 7,400,000 rows after cleaning.

I recommend converting the file to ***PARQUET*** as it drastically reduces its size and increases speed of commands.

# *USE* [Dashboard](https://blog.streamlit.io/crafting-a-dashboard-app-in-python-using-streamlit/)



## Main Questions:

1. How was crime change over the last 20 years?
2. Why have narcotic crimes decreased over time despite high drug use?
3. How is crime distributed across the city?
4. How did Covid affect crime rates?
5. Distribution of shootings?


### ***This is more of an exploritor notebook, I had created a 100,000 row file so data notebook can be tested, however results will vary from final product***

**Due to the nature of and size of the new dataframe not all graphs will render**

In [ ]:
import folium
from folium import plugins
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import geopandas as gpd

df = pd.read_csv("data/crime_data_sampled.csv")

Data is cleaned. All ** NaN ** values are dropped. The majority of NaN values are located in Community Area and would require a large ammount of resources for reverse GeoCoding, alternativly mapping bounding box and mapping the LongLat to the appropriate Community Area.

In [ ]:
#Data Cleaning
columns_to_delete = ['Case Number', 'Block', 'Ward', 'FBI Code', 'X Coordinate', 'Y Coordinate', 'Updated On', 'Location', 'Beat', 'Description','Location Description'] 
df = df.drop(columns=columns_to_delete)
df = df.dropna(subset=['District', 'Community Area','Latitude', 'Longitude' ])
df = df.astype({'District':'int', 'Community Area':'int'})
filtered_df = df

# Impact of Covid crime rates


In [ ]:
filtered_df['Date'] = pd.to_datetime(filtered_df['Date'], format='%m/%d/%Y %I:%M:%S %p')

#Filter with these dates: 
start_date = pd.to_datetime('1/03/2019', dayfirst=True)
end_date = pd.to_datetime('31/05/2023', dayfirst=True)

filtered_df_crime = filtered_df[(filtered_df['Date'] >= start_date) & (filtered_df['Date'] <= end_date)]
reports_per_month = filtered_df_crime.groupby([filtered_df_crime['Date'].dt.to_period('M')]).size()
reports_per_month_df = reports_per_month.reset_index(name='Count')
reports_per_month_df['Date'] = reports_per_month_df['Date'].dt.to_timestamp()

#TODO Annotate graph
plt.figure(figsize=(10, 6))
plt.plot(reports_per_month_df['Date'], reports_per_month_df['Count'], marker='o')
plt.title('Number of Reports per Month (21 March 2020 to 11 June 2021)')
plt.xlabel('Month')
plt.ylabel('Number of Reports')
plt.xticks(rotation=45)
plt.grid(True)
plt.tight_layout() 
specific_date = pd.to_datetime('2020-01')
count_at_specific_date = reports_per_month_df[reports_per_month_df['Date'] == specific_date]['Count'].values[0]
plt.annotate('Lockdown', xy=(specific_date, count_at_specific_date), xytext=(specific_date, count_at_specific_date+2700),
             arrowprops=dict(arrowstyle='-', linestyle=' ', color='red'), ha='left')
plt.axvline(x=specific_date, color='red', linestyle='--', linewidth=1)


# Adjust layout to make room for the rotated x-axis labels and ensure nothing is cut off
plt.tight_layout()
plt.show()


In [ ]:
# Convert the 'Date' column to datetime format if necessary
filtered_df['Date'] = pd.to_datetime(filtered_df['Date'])

# Define the lockdown period
lockdown_start = pd.Timestamp('2020-03-01')
lockdown_end = pd.Timestamp('2022-03-01')

# Split the data into three periods: before, during, and after the lockdown
before_lockdown = filtered_df[filtered_df['Date'] < lockdown_start]
during_lockdown = filtered_df[(filtered_df['Date'] >= lockdown_start) & (filtered_df['Date'] <= lockdown_end)]
after_lockdown = filtered_df[filtered_df['Date'] > lockdown_end]

# Function to calculate the average number of crimes per month_
def average_crimes_per_month(data):
    total_crimes = data.shape[0]
    num_months = ((data['Date'].max() - data['Date'].min()).days + 1) / 30  # Approximate to 30 days per month
    return total_crimes / num_months

# Calculate the average number of crimes per month for each period
avg_crimes_before = average_crimes_per_month(before_lockdown)
avg_crimes_during = average_crimes_per_month(during_lockdown)
avg_crimes_after = average_crimes_per_month(after_lockdown)

# Prepare data for plotting
percent_change_during = ((avg_crimes_during - avg_crimes_before) / avg_crimes_before) * 100
percent_change_after = ((avg_crimes_after - avg_crimes_during) / avg_crimes_during) * 100

# Print the results
print(f"Average Crimes per Month Before Lockdown: {avg_crimes_before:.2f}")
print(f"Average Crimes per Month During Lockdown: {avg_crimes_during:.2f}")
print(f"Average Crimes per Month After Lockdown: {avg_crimes_after:.2f}")
print(f"Percentage Change During Lockdown: {percent_change_during:.2f}%")
print(f"Percentage Change After Lockdown: {percent_change_after:.2f}%")

In [ ]:
before_lockdown_types = before_lockdown.groupby('Primary Type').size().reset_index(name='Before Counts')
during_lockdown_types = during_lockdown.groupby('Primary Type').size().reset_index(name='During Counts')
crime_comparison = pd.merge(before_lockdown_types, during_lockdown_types, on='Primary Type', how='outer').fillna(0)

# Calculate changes
crime_comparison['Change in Crimes'] = crime_comparison['During Counts'] - crime_comparison['Before Counts']
crime_comparison['Percentage Change'] = (crime_comparison['Change in Crimes'] / crime_comparison['Before Counts']) * 100
crime_comparison['Percentage Change'].replace([float('inf'), -float('inf')], float('nan'), inplace=True)

# Filter increases
increased_crimes = crime_comparison[crime_comparison['Change in Crimes'] > 0]
increased_crimes_sorted = increased_crimes.sort_values(by='Percentage Change', ascending=False)

# Display results
print(increased_crimes_sorted[['Primary Type', 'Percentage Change']])

# Visualisation of entire dataset in terms of crimes being commited.

In [ ]:
crime_counts = filtered_df['Primary Type'].value_counts().reset_index()
crime_counts.columns = ['Primary Type', 'Counts']

# Sorting the data for better visualization, if needed
crime_counts.sort_values(by='Counts', ascending=False, inplace=True)

# Creating the bar chart
plt.figure(figsize=(12, 8))  # Set the figure size for better readability
plt.bar(crime_counts['Primary Type'], crime_counts['Counts'], color='blue')  # Create a bar plot
plt.xlabel('Type of Crime')  # Label for the x-axis
plt.ylabel('Number of Incidents')  # Label for the y-axis
plt.title('All Crimes Committed by Type')  # Title of the plot
plt.xticks(rotation=90)  # Rotate x-axis labels for better readability
plt.tight_layout()  # Adjust layout to make room for the rotated x-axis labels

# Show the plot
plt.show()

# Crime distrubtion across 24 hours.

We can clearly see a slump in the evening hours but an overall steady crime rate. The spike in crime reports around 12 oclock seem to be attributed to how the logging software is used. The same spike can be seen at 12:00.


In [ ]:
chicago_map = folium.Map([41.85, -87.68], zoom_start=10.6)

#A sample of the total amount of reports is taken at random as plotting all points is not possible. 1/14 of total reports represented at random.
df_500k = filtered_df.sample(n=500_000, random_state=42)
points_of_interest = df_500k[['Latitude', 'Longitude']]
chicago_map.add_child(plugins.HeatMap(points_of_interest,min_opacity=0.35, blur = 15, radius=0))

In [ ]:
#filtered_df = df[(df['Primary Type'] == 'HOMICIDE') | (df['Crime'] == 'assault')]
filtered_homicide = filtered_df[(filtered_df['Primary Type'] == 'HOMICIDE')]
filtered_homicide_pos = filtered_homicide[['Longitude', 'Latitude']]

In [ ]:
#Homicides across the city not displaying correctly as heatmap. #Prio Fix
locations = filtered_homicide[['Latitude', 'Longitude']].values.tolist()
test_map = folium.Map(location=[41.85, -87.68], zoom_start= 10.5)

for location in locations:
    folium.CircleMarker(location=location, radius=1).add_to(test_map)
    
test_map

In [ ]:
filtered_df['Year'] = pd.to_numeric(filtered_df['Year'], errors='coerce')

# Excluding 2001 & 2024 as the data for those years is not complete.
data_filtered = filtered_df[(filtered_df['Year'] != 2024) & (filtered_df['Year'] != 2001)]
crime_counts_per_year = data_filtered.groupby(['Year', 'Primary Type']).size().reset_index(name='Counts')
top_5_crimes_per_year = crime_counts_per_year.groupby('Year').apply(
    lambda x: x.nlargest(5, 'Counts')
).reset_index(drop=True)
top_5_crimes_per_year

In [ ]:
plot_data_crimes = top_5_crimes_per_year.pivot(index='Year', columns='Primary Type', values='Counts').fillna(0)

# Plot
plt.figure(figsize=(14, 8))
sns.lineplot(data=plot_data_crimes, dashes=False, markers=True, linewidth=2.5, alpha=0.85)

plt.title('Top 5 Committed Crimes per Year Over the Last 20 Years', fontsize=16)
plt.xlabel('Year', fontsize=14)
plt.ylabel('Number of Incidents', fontsize=14)
plt.xticks(rotation=45)
plt.legend(title='Crime Type')
plt.grid(True, which='both', linestyle='--', linewidth=0.5)
plt.tight_layout()

plt.show()

In [ ]:
#Using IUCR to target all NARCOTIC crimes. IUCR data located in data file. Manually adjusted.
filtered_df['IUCR'] = pd.to_numeric(filtered_df['IUCR'], errors='coerce')
narcotics_data = filtered_df[(filtered_df['IUCR'] >= 1811) & (filtered_df['IUCR'] <= 2170)]
narcotics_data['Date'] = pd.to_datetime(narcotics_data['Date'])
narcotics_data['Year'] = narcotics_data['Date'].dt.year
narcotics_counts_per_year = narcotics_data.groupby('Year').size()

#Plot
plt.figure(figsize=(10, 6))
narcotics_counts_per_year.plot(kind='line', marker='o', linestyle='-', color='green')
plt.title('Narcotics Crimes Over Time')
plt.xlabel('Year')
plt.ylabel('Number of Narcotics Crimes')
plt.grid(True)
plt.tight_layout()

plt.show()

In [ ]:

#Mapping
map_chiacgo = folium.Map(location=[41.8781, -87.6298], zoom_start=10)
geo_boundary_chicago = gpd.read_file('data/chicago_boundaries.geojson')

#Data handling
total_crime_count = filtered_df.groupby('Community Area').size().reset_index(name='Crimes')
geo_boundary_chicago['community'] = geo_boundary_chicago['area_num_1'].astype(int)
total_crime_count['Community Area'] = total_crime_count['Community Area'].astype(int)
geo_boundary_chicago_merged = geo_boundary_chicago.merge(total_crime_count, left_on='community', right_on='Community Area', how='left')

folium.Choropleth(
    geo_data=geo_boundary_chicago_merged,
    name="choropleth",
    data=geo_boundary_chicago_merged,
    columns=["community", "Crimes"],
    key_on="feature.properties.area_num_1",
    fill_color="PuBu",
    fill_opacity=0.7,
    line_opacity=0.2,
    legend_name="Crimes"
).add_to(map_chiacgo)

folium.LayerControl().add_to(map_chiacgo)

map_chiacgo


#geo_boundary_chicago_merged.to_csv('data/data_cache/geo_please.csv')

In [ ]:
geo_boundary_chicago_merged.dropna(subset=['community', 'Crimes'], inplace=True)
geo_boundary_chicago_merged = geo_boundary_chicago_merged[geo_boundary_chicago_merged.geometry.notnull()]

# Reset the index
geo_boundary_chicago_merged.reset_index(drop=True, inplace=True)

In [ ]:
geo_json_data = geo_boundary_chicago_merged.to_json()



map_chicago = folium.Map(location=[41.8781, -87.6298], zoom_start=10)
# Update the Choropleth creation part in your function
folium.Choropleth(
    geo_data=geo_json_data,
    name='choropleth',
    data=geo_boundary_chicago_merged,
    columns=['community', 'Crimes'],
    key_on='feature.properties.area_num_1',  # Adjust this line if necessary
    fill_color='PuBu',
    fill_opacity=0.7,
    line_opacity=0.2,
    legend_name='Crimes'
).add_to(map_chicago)

map_chicago

In [ ]:
map_chiacgo_2 = folium.Map(location=[41.8781, -87.6298], zoom_start=10)
geo_boundary_chicago = gpd.read_file('data/chicago_boundaries.geojson')

homicides_data = filtered_df[filtered_df['Primary Type'].str.upper() == 'HOMICIDE']
homicide_counts = homicides_data.groupby('Community Area').size().reset_index(name='Homicide Crimes')
geo_boundary_chicago['community'] = geo_boundary_chicago['area_num_1'].astype(int)
homicide_counts['Community Area'] = homicide_counts['Community Area'].astype(int)
geo_boundary_chicago_merged = geo_boundary_chicago.merge(homicide_counts, left_on='community', right_on='Community Area', how='left')


folium.GeoJson(
    geo_boundary_chicago_merged.to_json(),
    name="Homicides",
    style_function=lambda feature: {
        'color': 'grey', 
        'weight': 1, 
        'fillColor': 'PuBu',
        'fillOpacity': 0.8,
    },
    tooltip=folium.GeoJsonTooltip(
        fields=['community', 'Homicide Crimes'],
        aliases=['Community Area:', 'Homicide Crimes:'],
        localize=True
    )
).add_to(map_chiacgo_2)

folium.LayerControl().add_to(map_chiacgo_2)

map_chiacgo_2

In [ ]:
austin_crimes = filtered_df[filtered_df['Community Area'] == 25].shape[0]
total_crimes = filtered_df.shape[0]
percentage_austin_crimes = (austin_crimes / total_crimes) * 100

austin_crimes, total_crimes, percentage_austin_crimes

In [ ]:
homicides_data = filtered_df[filtered_df['Primary Type'].str.upper() == 'HOMICIDE']
austin_homicides = homicides_data[homicides_data['Community Area'] == 25].shape[0]
total_homicides = homicides_data.shape[0]
percentage_austin_homicides = (austin_homicides / total_homicides) * 100

print(f"Austin Homicides: {austin_homicides}")
print(f"Total Homicides: {total_homicides}")
print(f"Percentage of Homicides in Austin: {percentage_austin_homicides}%")

In [ ]:
# Filter for homicides
homicides = df[df["Primary Type"] == "HOMICIDE"]

# Group by year and count incidents
homicides_per_year = homicides.groupby("Year").size()

# Calculate average homicides per year
average_homicides = homicides_per_year.mean()

# Output results
print("Average homicides per year:", round(average_homicides, 2))
print("\nHomicides per year:\n", homicides_per_year)

In [ ]:
# Filter for homicides
homicides = df[df["Primary Type"] == "HOMICIDE"]

# Count total deaths
total_deaths = len(homicides)

# Average population of Chicago (over 25 years)
average_population = 2_700_000

# Calculate deaths per 100,000 people
deaths_per_100k = (total_deaths / average_population) * 100_000

# Output
print("Total homicides (deaths):", total_deaths)
print("Deaths per 100,000 people:", round(deaths_per_100k, 2))

In [ ]:
# Calculate crime count by area
crime_counts = df.groupby("Community Area").size()

# Exclude invalid area 0
crime_counts = crime_counts[crime_counts.index != 0]

# Get top and bottom 3 areas
top_3 = crime_counts.sort_values(ascending=False).head(5).index.tolist()
bottom_3 = crime_counts.sort_values().head(5).index.tolist()

# Filter and group by crime type
top_crimes = df[df["Community Area"].isin(top_3)].groupby("Primary Type").size().sort_values(ascending=False)
bottom_crimes = df[df["Community Area"].isin(bottom_3)].groupby("Primary Type").size().sort_values(ascending=False)

# Align for comparison
bottom_aligned = bottom_crimes.reindex_like(top_crimes).fillna(0)

# Plot comparison
plt.figure(figsize=(14, 7))
width = 0.4
indices = range(len(top_crimes))

plt.bar(indices, top_crimes.values, width=width, label='Top 3 Areas')
plt.bar([i + width for i in indices], bottom_aligned.values, width=width, label='Bottom 3 Areas')
plt.title('Crime Type Comparison: Top 3 vs Bottom 3 Community Areas')
plt.xlabel('Crime Type')
plt.ylabel('Number of Incidents')
plt.xticks([i + width / 2 for i in indices], top_crimes.index, rotation=45, ha='right')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Tourist community area IDs (based on Chicago official list)
tourist_area_ids = [8, 32, 7, 6, 77, 61, 33, 58, 34, 38, 28, 24]

# Classify violent crimes
violent_crimes = {
    "HOMICIDE", "BATTERY", "ASSAULT", "ROBBERY", "CRIM SEXUAL ASSAULT", "KIDNAPPING", "ARSON"
}

# Label area and crime category
df["Tourist Area"] = df["Community Area"].apply(
    lambda x: "Tourist" if x in tourist_area_ids else "Non-Tourist"
)
df["Crime Category"] = df["Primary Type"].apply(
    lambda x: "Violent" if x in violent_crimes else "Non-Violent"
)

# Group and compare
tourist_vs_non = df.groupby(["Tourist Area", "Crime Category"]).size().unstack()

# Plot
tourist_vs_non.plot(kind='bar', figsize=(10, 6))
plt.title("Violent vs Non-Violent Crimes: Tourist vs Non-Tourist Areas")
plt.xlabel("Area Type")
plt.ylabel("Number of Incidents")
plt.xticks(rotation=0)
plt.legend(title="Crime Category")
plt.tight_layout()
plt.show()


In [ ]:
# Calculate total crimes by category for both area types
total_crimes_tourist = tourist_vs_non.loc["Tourist"].sum()
total_crimes_non_tourist = tourist_vs_non.loc["Non-Tourist"].sum()

# Calculate percentage difference
percentage_difference = ((total_crimes_non_tourist - total_crimes_tourist) / total_crimes_tourist) * 100

print(f"Percentage difference in crime: {percentage_difference:.2f}%")

